# 05 — Lead-Lag Analysis (Full Universe)

**Purpose:** Cross-correlation and Granger causality analysis across the full asset universe to identify which assets consistently move before others. This is alpha signal work — not just hedging.

**Run manually** when exploring new signals. Not part of the automated pipeline.

**Output:** Updates `correlation_state.json` with full-universe lead-lag pairs.

**Sections:**
1. Setup & data
2. Cross-correlation lead-lag (fast)
3. Granger causality (selective — top pairs only)
4. Visualisations
5. Export

In [ ]:
import sys, os, json, logging
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import grangercausalitytests

NOTEBOOK_DIR   = os.path.abspath('')
RESEARCH_ROOT  = os.path.join(NOTEBOOK_DIR, '..')
PORTFOLIO_ROOT = os.path.join(RESEARCH_ROOT, '..', 'portfolio')
sys.path.insert(0, RESEARCH_ROOT)
sys.path.insert(0, PORTFOLIO_ROOT)

from src.config import (
    ASSET_UNIVERSE, BENCHMARK_TICKER, LOOKBACK_DAYS,
    MAX_LAG_DAYS, GRANGER_PVALUE_THRESHOLD, OUTPUT_CORRELATION,
)
from src.correlation import compute_lead_lag
from src.data_loader import fetch_historical, calculate_log_returns, fetch_fx_rate, convert_usd_prices_to_eur, load_ledger

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white','axes.grid':True,'grid.alpha':0.3,'font.size':11})
print('✅ Imports OK')

In [ ]:
CACHE_PATH  = os.path.join(PORTFOLIO_ROOT, 'data', 'historical_prices.csv')
LEDGER_PATH = os.path.join(PORTFOLIO_ROOT, 'data', 'ledger.csv')

prices_raw  = fetch_historical(ASSET_UNIVERSE, LOOKBACK_DAYS, CACHE_PATH)
usd_eur     = fetch_fx_rate('USD', 'EUR')
prices      = convert_usd_prices_to_eur(prices_raw, usd_eur)
log_returns = calculate_log_returns(prices)
holdings, _ = load_ledger(LEDGER_PATH)
held_tickers = [t for t in holdings.keys() if t in log_returns.columns]

# For full-universe lead-lag, focus on liquid, high-coverage tickers
# (drop any with >5% missing days in the lookback)
coverage = (log_returns.notna().sum() / len(log_returns))
liquid_tickers = coverage[coverage > 0.95].index.tolist()
print(f'Full universe  : {len(log_returns.columns)} tickers')
print(f'Liquid (>95%)  : {len(liquid_tickers)} tickers')
print(f'Held tickers   : {held_tickers}')

## 2. Cross-correlation lead-lag

For every pair in the liquid universe, tests lags 1–5 days. Reports pairs where a lagged correlation is meaningfully stronger than the contemporaneous one (threshold: +0.05).

⚠️ On 50+ tickers this may take 2–5 minutes.

In [ ]:
# Run on liquid tickers (capped at 40 for speed — edit to taste)
TICKER_CAP = 40
run_tickers = liquid_tickers[:TICKER_CAP]
print(f'Running lead-lag on {len(run_tickers)} tickers (cap={TICKER_CAP}), max lag={MAX_LAG_DAYS}d...')

ll_results = compute_lead_lag(log_returns, run_tickers, max_lag=MAX_LAG_DAYS)
ll_df = pd.DataFrame(ll_results)

print(f'\n✅ {len(ll_df)} significant lead-lag pairs found')
if not ll_df.empty:
    display(ll_df.head(30))

## 3. Granger causality — top cross-correlation pairs

Granger causality tests whether past values of series A help predict series B *beyond* what B's own past values predict. It's a stronger claim than cross-correlation.

We only run Granger on the top 20 pairs by `lead_strength` — it's computationally intensive.

In [ ]:
granger_results = []

if ll_df.empty:
    print('No lead-lag pairs to test with Granger.')
else:
    top_pairs = ll_df.head(20)
    print(f'Running Granger causality on top {len(top_pairs)} pairs (max lag={MAX_LAG_DAYS})...')

    for _, row in top_pairs.iterrows():
        leader, follower = row['leader'], row['follower']
        try:
            pair_data = pd.concat([
                log_returns[follower].rename('follower'),
                log_returns[leader].rename('leader'),
            ], axis=1).dropna()

            # grangercausalitytests returns results per lag
            gc = grangercausalitytests(pair_data[['follower', 'leader']], maxlag=MAX_LAG_DAYS, verbose=False)

            # Pick the best lag (lowest F-test p-value)
            best_lag, best_pval = None, 1.0
            for lag, res in gc.items():
                pval = res[0]['ssr_ftest'][1]  # F-test p-value
                if pval < best_pval:
                    best_pval = pval
                    best_lag  = lag

            granger_results.append({
                'leader':          leader,
                'follower':        follower,
                'best_lag_days':   best_lag,
                'granger_pvalue':  round(best_pval, 5),
                'granger_sig':     best_pval < GRANGER_PVALUE_THRESHOLD,
                'xcorr_strength':  row['lead_strength'],
            })
        except Exception as e:
            logging.warning(f'Granger failed for {leader}→{follower}: {e}')

    granger_df = pd.DataFrame(granger_results).sort_values('granger_pvalue')
    n_sig = granger_df['granger_sig'].sum()
    print(f'\n✅ Granger complete | {n_sig}/{len(granger_df)} pairs statistically significant (p<{GRANGER_PVALUE_THRESHOLD})')
    display(granger_df)

## 4. Visualisations

In [ ]:
if not ll_df.empty:
    top20 = ll_df.head(20).copy()
    top20['pair'] = top20['leader'] + ' → ' + top20['follower']

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Lead strength
    axes[0].barh(top20['pair'][::-1], top20['lead_strength'][::-1], color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Lead strength (Δ corr vs lag 0)')
    axes[0].set_title('Top 20 lead-lag pairs (cross-correlation)')

    # Lag distribution
    lag_counts = top20['lag_days'].value_counts().sort_index()
    axes[1].bar(lag_counts.index, lag_counts.values, color='coral', edgecolor='white')
    axes[1].set_xlabel('Lead lag (days)')
    axes[1].set_ylabel('Number of pairs')
    axes[1].set_title('Distribution of lead lag (days)')
    axes[1].set_xticks(range(1, MAX_LAG_DAYS + 1))

    plt.tight_layout()
    plt.show()

    # Cross-corr at best lag vs lag 0
    fig, ax = plt.subplots(figsize=(10, 5))
    top10 = ll_df.head(10)
    x = np.arange(len(top10))
    ax.bar(x - 0.2, top10['corr_at_0'],   0.35, label='Corr at lag 0', color='#aaa', edgecolor='white')
    ax.bar(x + 0.2, top10['corr_at_lag'],  0.35, label='Corr at best lag', color='steelblue', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r['leader']}→{r['follower']}" for _, r in top10.iterrows()], rotation=30, ha='right')
    ax.set_title('Lag 0 vs best-lag correlation — top 10 pairs')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Export

Updates `correlation_state.json` with full-universe lead-lag and Granger results.

In [ ]:
existing = {}
if os.path.exists(OUTPUT_CORRELATION):
    with open(OUTPUT_CORRELATION, 'r') as f:
        existing = json.load(f)

existing.update({
    'leadlag_generated_at':   datetime.now().isoformat(),
    'leadlag_universe_size':  len(run_tickers),
    'lead_lag_pairs_full':    ll_results,
    'granger_results':        granger_results,
    'granger_significant':    [r for r in granger_results if r.get('granger_sig')],
})

with open(OUTPUT_CORRELATION, 'w', encoding='utf-8') as f:
    json.dump(existing, f, indent=2, default=str)

print(f'✅ Exported → {OUTPUT_CORRELATION}  ({os.path.getsize(OUTPUT_CORRELATION)/1024:.1f} KB)')
print(f'   {len(ll_results)} cross-corr pairs + {len(granger_results)} Granger tests added.')